In [ ]:
# Training Kata dengan Model BiGRU
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# ==========================================================
# 1. KONFIGURASI & HYPERPARAMETER
# ==========================================================
DATA_PATH = "dataset_npy"
MODEL_OUTPUT_PATH = "model_gru_kata.pth"

INPUT_SIZE = 104    # 104 fitur landmark per frame
SEQ_LENGTH = 30    # 30 frame per video
HIDDEN_SIZE = 128   # jumlah neuron di hidden layer GRU
NUM_LAYERS = 2     # jumlah tumpukan layer GRU

# --- PERUBAHAN OTOMATIS ---
# Kita pindahkan penetapan NUM_CLASSES setelah me-load dataset agar dinamis
# ==========================================================

BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 0.001

# Gunakan GPU jika tersedia, jika tidak pakai CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# ==========================================================
# 2. LOAD DATASET & PREPARATION
# ==========================================================
print("Memuat data npy...")
X = np.load(os.path.join(DATA_PATH, "X.npy"))
y = np.load(os.path.join(DATA_PATH, "y.npy"))

print(f"Shape X asli: {X.shape}") # (total_video, 30, 104)
print(f"Shape y asli: {y.shape}") # (total_video,)

# --- PERUBAHAN OTOMATIS UNTUK TOTAL KELAS ---
# Mencari nilai unik tertinggi di dalam array y untuk menetapkan jumlah kelas asli dataset
unique_classes = np.unique(y)
NUM_CLASSES = len(unique_classes)
print(f"[INFO] Terdeteksi jumlah kelas dari data y.npy: {NUM_CLASSES} kelas (Indeks: {unique_classes.min()}-{unique_classes.max()})")

# Bagi data: 80% Training, 20% Validasi/Testing
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ubah ke Tensor PyTorch dan sesuaikan tipe datanya
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

# Bungkus ke dalam DataLoader agar bisa ditraining per batch
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================================
# 3. DEFINISI ARSITEKTUR ARSITEKTUR BI-GRU
# ==========================================================
class IsyaratBiGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(IsyaratBiGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Bidirectional=True membuat output berdimensi hidden_size * 2
        self.bigru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )
        
        # Fully Connected Layer untuk klasifikasi akhir
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        out, _ = self.bigru(x)
        
        # Ambil output dari frame terakhir (sekuensial selesai)
        out = self.fc(out[:, -1, :])
        return out

# Inisialisasi model menggunakan NUM_CLASSES yang sudah diperbarui otomatis
model = IsyaratBiGRU(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)

# Loss dan Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==========================================================
# 4. TRAINING LOOP
# ==========================================================
print("\nMemulai proses training...")
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass dan optimasi
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100
    
    # Evaluasi pada data validasi
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"| Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% "
          f"| Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")
    
    # Simpan model terbaik berdasarkan akurasi validasi tertinggi
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), MODEL_OUTPUT_PATH)
        print(f"  --> Model terbaik disimpan dengan Akurasi Val: {best_val_acc:.2f}%")

print(f"\n[OK] Training selesai! Model terbaik disimpan di '{MODEL_OUTPUT_PATH}'")